# Detection EPI chantier - YOLO (Colab / Kaggle)

Pipeline complet : conversion XML -> YOLO, entrainement, evaluation, inference.

**Avant de lancer :**
- Colab : `Execution > Modifier le type d'execution > T4 GPU`
- Kaggle : Settings > Accelerator > GPU T4 x2 (ou P100), et active Internet dans Settings
- Uploade ton dataset zippe (ex. `dataset.zip` contenant `training/` et `testing/`)

## 0. Environnement

In [ ]:
!pip install -q ultralytics
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| ultralytics', ultralytics.__version__)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. Recuperer le code + le dataset

Option A (le plus simple) : upload manuel du zip du projet `ppe-yolo` et du dataset.
Option B (Colab) : monter Google Drive.
Option C (Kaggle) : ajouter le dataset via *Add Data* -> il apparait dans `/kaggle/input/...`

In [ ]:
# --- Option A : upload
# from google.colab import files
# files.upload()   # dataset.zip, puis :
# !unzip -q dataset.zip -d /content/raw

# --- Option B : Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q '/content/drive/MyDrive/dataset.zip' -d /content/raw

# --- Option C : Kaggle
# !ls /kaggle/input

import os
os.makedirs('/content/raw', exist_ok=True)
!ls -R /content/raw | head -20

## 2. Conversion XML -> YOLO

On utilise `scripts/xml_to_yolo.py` du repo (adapte le chemin). Le split
train/val (80/20) est fait ici de facon reproductible, **test/ reste intouche**.

In [ ]:
# Si tu as clone le repo, le script est deja la :
# !git clone https://github.com/TON_USER/ppe-yolo.git
# !python ppe-yolo/scripts/xml_to_yolo.py --src /content/raw --dst /content/datasets/ppe

# Sinon, colle ici la fonction de conversion (voir le script du repo).
!python ppe-yolo/scripts/xml_to_yolo.py --src /content/raw --dst /content/datasets/ppe

In [ ]:
!python ppe-yolo/scripts/dataset_stats.py --data /content/datasets/ppe
!cat /content/datasets/ppe/ppe.yaml

## 3. Entrainement

`yolov8n.pt` = 3.2 M parametres, pre-entraine sur COCO -> convergence rapide, parfait
quand le dataset est petit (1080 images). Le modele *custom* `yolov8n-ppe.yaml`
(~1.3 M) est a reserver au CPU.

Sur Colab/Kaggle : `imgsz=640, batch=16, epochs=100` (~30-45 min sur T4).

In [ ]:
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n.pt')   # ou 'ppe-yolo/configs/yolov8n-ppe.yaml' pour from scratch

results = model.train(
    data='/content/datasets/ppe/ppe.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=8,
    project='/content/runs', name='ppe_v1', exist_ok=True,
    patience=25, cos_lr=True,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, close_mosaic=15,
    save_period=10,   # sauvegarde intermediaire (Colab coupe parfois)
)

## 4. Courbes d'apprentissage

A regarder : `results.png` (box_loss / cls_loss doivent descendre, mAP50 monter),
`confusion_matrix.png`, `val_batch*_pred.jpg`.

In [ ]:
from IPython.display import Image, display
display(Image('/content/runs/ppe_v1/results.png', width=900))
display(Image('/content/runs/ppe_v1/confusion_matrix.png', width=600))

## 5. Evaluation finale sur le TEST set

A ne faire qu'une fois, quand la validation te satisfait.

In [ ]:
best = YOLO('/content/runs/ppe_v1/weights/best.pt')
m = best.val(data='/content/datasets/ppe/ppe.yaml', split='test', imgsz=640, conf=0.001, iou=0.6)
print('mAP50   :', round(m.box.map50, 4))
print('mAP50-95:', round(m.box.map, 4))
for i, idx in enumerate(m.ap_class_index):
    print(f'{best.names[int(idx)]:<14} P={m.box.p[i]:.3f} R={m.box.r[i]:.3f} mAP50={m.box.ap50[i]:.3f}')

## 6. Inference sur des images / export

Export ONNX -> utile pour tourner sur CPU portable sans torch (2-3x plus rapide) puis
`model = YOLO('best.onnx')`.

In [ ]:
best.predict('/content/datasets/ppe/images/test', imgsz=640, conf=0.25, save=True,
             project='/content/runs', name='pred_test')

# export CPU-friendly
best.export(format='onnx', imgsz=640, dynamic=False, simplify=True)
!ls -lh /content/runs/ppe_v1/weights/

In [ ]:
# Telecharger les resultats (Colab)
# !zip -r /content/ppe_results.zip /content/runs/ppe_v1
# from google.colab import files; files.download('/content/ppe_results.zip')

## 7. (Optionnel) Ameliorer les resultats

Par ordre de cout/benefice :
1. Plus d'epochs (150) + `imgsz=640` : gratuit, souvent +3-5 mAP.
2. `yolov8s.pt` : 11 M params, +2-4 mAP, ~2x plus lent. A comparer.
3. Corriger les annotations (les casques mal annotes = plafond de verite).
4. `copy_paste=0.3` si `vest` est sous-representee.
5. Test-time augmentation : `best.val(..., augment=True)` (+1-2 mAP, 2x plus lent).